# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
This dataset is described via a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name + ': ' + metadata.description)

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

We will enumerate all record sets and their fields by `@id`.

In [ ]:
# List record sets and fields with their IDs
record_sets = dataset.metadata.recordSet

if not record_sets:
    print('No record sets found in metadata.')
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        if fields:
            for f in fields:
                print(f"  Field @id: {f['@id']}, name: {f.get('name', '<no name>')}")
        else:
            print('  No fields found in this record set.')

## 3. Data Extraction

Load data from specific record set(s) into a DataFrame for analysis.

Use record set and field `@id`s as referenced above.

In [ ]:
# If record sets are present, extract their data
dataframes = {}
extracted_record_sets = []

if record_sets:
    for rs in record_sets:
        record_set_id = rs['@id']
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                extracted_record_sets.append(record_set_id)
                print(f"Loaded RecordSet {record_set_id}, shape: {df.shape}, columns: {df.columns.tolist()}")
                print(df.head())
            else:
                print(f"No records found for RecordSet {record_set_id}")
        except Exception as e:
            print(f"Error loading records from RecordSet {record_set_id}: {e}")

else:
    print('No record sets available to extract.')

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by attributes.

We will select a numeric field (e.g., total score from a survey) from the first available record set, filter values above a threshold, normalize, and group data.

In [ ]:
# Choose the first extracted record set for EDA
if extracted_record_sets:
    rs_id = extracted_record_sets[0]
    df = dataframes[rs_id]
    # Try to find a numeric field (e.g., 'phq9_total_score', 'gad7_total_score', 'psq_total_score')
    numeric_candidates = [col for col in df.columns if 'score' in col or df[col].dtype in ['int64', 'float64']]

    numeric_field = numeric_candidates[0] if numeric_candidates else df.columns[0]
    # Display field selected
    print(f"Analyzing numeric field: {numeric_field}")

    # Filtering records
    threshold = 10
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by an attribute field, try 'level_of_education', 'gender', or first categorical found
        candidate_group_fields = [col for col in df.columns if 'level_of_education' in col or 'gender' in col or df[col].dtype == 'object']
        group_field = candidate_group_fields[0] if candidate_group_fields else numeric_field
        print(f"Grouping by: {group_field}")
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
    else:
        print(f"{numeric_field} is not numeric, skipping EDA.")
else:
    print('No dataframes available for EDA.')

## 5. Visualization

Visualize numeric field distributions and group means using matplotlib and seaborn.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only run visualization if previous EDA was successful
if extracted_record_sets and 'filtered_df' in locals():
    # Plot histogram of normalized scores
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, bins=15)
    plt.title(f"Histogram of Normalized {numeric_field} (Filtered)")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Count")
    plt.show()

    # Plot group mean
    if 'grouped_df' in locals() and group_field in grouped_df.columns:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field} (Filtered)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No filtered data available for visualization.')

## 6. Conclusion

- Using `mlcroissant`, we loaded and explored the FAIR² Mental Health Survey Dataset, referencing all entities by their `@id` for reproducibility and clarity.
- We extracted survey responses from Kilifi County, reviewed available record sets and fields, filtered and normalized scores such as PHQ-9, GAD-7, or PSQ, and visualized their distributions.
- This notebook demonstrates how to utilize Croissant metadata standards and `mlcroissant`'s programmatic interface for AI-ready survey data analysis.

**Next steps:**
- Consider deeper analyses of field relationships (e.g., regression, clustering).
- Incorporate additional visualizations and statistical tests.
- Explore other record sets and documentation to further understand the dataset structure and survey variables.